### Loading the documnets

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import DirectoryLoader

try:

    doc_loader = DirectoryLoader(
        path='../data/',
        glob='**/*.docx',
        loader_cls=Docx2txtLoader,
        show_progress=True
    )

    pdf_loader = DirectoryLoader(
        path='../data/',
        glob='**/*.pdf',
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )

except Exception as e:
    print (f"Error while loding documnets {e}")

C:\Users\praju\AppData\Local\Temp\ipykernel_33076\228811747.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\praju\Desktop\langchain-new\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading the documnets...")
pdf_data = pdf_loader.load() 
doc_data =  doc_loader.load()
print("Documnets loaded succesfully...")
print(f'Length of the loaded PDF documents: {len(pdf_data)}')
print(f'Length of the loaded DOCX documents: {len(doc_data)}')

Loading the documnets...


100%|██████████| 1/1 [00:00<00:00, 32.36it/s]

Documnets loaded succesfully...
Length of the loaded PDF documents: 1
Length of the loaded DOCX documents: 1


### Creating output structure

In [3]:
from pydantic import BaseModel, Field

class resume(BaseModel):
    ats_score:int = Field(description="Resume overall ATS score")
    grammer_score:int = Field(description="Resume overall Grammer score")
    skills_found : list = Field(description="All the skill found in the resume")
    missing_skills : list = Field(description="Suggest importent skills, besides the skills already exists")
    interview_questions : list = Field(description="Provide a list of interview question based on the resume")
    suggestion : list = Field(description="Suggest improvement for the uploaded resume")

### Loading LLM model and creating Agent

In [4]:
from langchain.messages import SystemMessage
from langchain.agents import create_agent
from langchain_fireworks import ChatFireworks
from dotenv import load_dotenv
import os 

load_dotenv()

system_msg = SystemMessage(
    """
you are an ATS + HR expert resume reviewer 
go trought all the resume in depth and  evaluate ATS score, 
grammar, skills, missing skills, interview questions, and 
suggestions based on the resume 
"""
)
print('Loading LLM model...')
llm = ChatFireworks(
    model='accounts/fireworks/models/kimi-k2p5',
    api_key=os.getenv('FIREWORK_API_KEY')
)

llm_with_output = llm.with_structured_output(resume)

llm_info = {
    "name": llm.profile["name"],
    "max_input_tokens": llm.profile["max_input_tokens"],
    "max_output_tokens": llm.profile["max_output_tokens"],
    "release_date": llm.profile['release_date'],
    "last_updated": llm.profile['last_updated']
}

print(f'LLM model loaded: {llm_info}')

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000018F19C13550>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x0000018F19C441C0>


Loading LLM model...
LLM model loaded: {'name': 'Kimi K2.5', 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'release_date': '2026-01-27', 'last_updated': '2026-01-27'}
